# Train BSRNN + TF-Map — `mid` split, Kaggle GPU

Everything needed to train. Nothing else: no eval harness, no live-model metric,
no data generation, no analysis.

## Before you run

1. **Settings -> Accelerator -> GPU** (T4 or P100).
2. **Add data ->** both datasets: the one from `kaggle_data.zip` (2.7 GB, upload
   once) and the one from `kaggle_code.zip` (small, re-upload whenever the code
   changes). They are separate so a one-line fix never costs a 2.7 GB upload.
3. Set `EPOCHS` in the next cell. Run with `EPOCHS = 2` first to get a measured
   seconds/epoch, then decide — Kaggle kills a GPU session at 12 h and
   `history.csv` is only written when training *finishes*. The checkpoint is
   saved on every val improvement, so a kill costs the curve, not the weights.

## If the session dies before it finishes

`history.csv` is only written when training completes, but the training cell
prints one CSV row per epoch as it goes. Select that block (header included),
paste it into `history.csv`, and the curve is intact. The same rows are also
appended live to `/kaggle/working/results/history_live.csv`.

The checkpoint is saved on every val improvement, so the weights survive too.

## To resume in a later session

Save Version, then add this notebook's own output as a dataset input and set
`RESUME_FROM` to the `model_mid.pt` inside it. `train.py` refuses to resume
across a config change, so do not edit the knobs between sessions.

In [ ]:
# ============================== KNOBS ==============================
EPOCHS      = 2       # start at 2 to measure, then raise. 12 h GPU cap.
BATCH_SIZE  = 12      # the config's intended value; 3 was a laptop-memory limit
NUM_WORKERS = 4       # 0 starves the GPU: 3 windowed wav reads per crop
RESUME_FROM = None    # e.g. "/kaggle/input/prev-run/model_mid.pt"

DATA_DIR    = None    # None = auto-discover under /kaggle/input
CODE_DIR    = None    # None = auto-discover under /kaggle/input
# ===================================================================

WORK = "/kaggle/working"
REPO = f"{WORK}/repo"          # writable copy of the code; /kaggle/input is not
OUT  = f"{WORK}/models"
RES  = f"{WORK}/results"

In [ ]:
# --- environment + locate the bundle -------------------------------------
import os, sys, shutil, subprocess, json
from pathlib import Path

import torch
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  device: {torch.cuda.get_device_name(0)}  "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("  NO GPU -- set Settings -> Accelerator -> GPU. Refusing to train on CPU here.")

# soundfile is the only dep Kaggle sometimes lacks; everything else is preinstalled
try:
    import soundfile
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile"], check=True)
    import soundfile
print(f"soundfile {soundfile.__version__}")

ROOTS = sorted(Path("/kaggle/input").glob("*")) if Path("/kaggle/input").exists() else []

def find(marker, override, what):
    """Locate the input dir containing `marker`. Kaggle sometimes nests the zip
    contents one level down, so check each dataset root and its subdirectories."""
    if override:
        return Path(override)
    for r in ROOTS:
        for cand in [r, *sorted(p for p in r.glob("*") if p.is_dir())]:
            if (cand / marker).exists():
                return cand
    raise SystemExit(f"Could not find the {what} dataset (looked for {marker}). "
                     f"Saw under /kaggle/input: {[r.name for r in ROOTS]}")

DATA = find("data/manifests/mid_train.csv", DATA_DIR, "data")
CODE = find("scripts/train.py", CODE_DIR, "code")
print(f"data: {DATA}\ncode: {CODE}")
for base, rel in [(CODE, "experiments/configs/bsrnn_baseline.yaml"),
                  (CODE, "src/models/bsrnn.py"),
                  (DATA, "data/manifests/mid_val.csv")]:
    assert (base / rel).exists(), f"missing {rel} under {base}"
print("  contents verified")

In [ ]:
# --- stage the code somewhere writable, then write the derived config -----
# /kaggle/input is read-only and src.run_log writes repo_root/docs/run_times.md,
# so the code cannot run in place.
import yaml

if Path(REPO).exists():
    shutil.rmtree(REPO)
shutil.copytree(CODE, REPO, ignore=shutil.ignore_patterns("data"))
Path(OUT).mkdir(parents=True, exist_ok=True)
Path(RES).mkdir(parents=True, exist_ok=True)

cfg_path = Path(REPO) / "experiments/configs/bsrnn_baseline.yaml"
cfg = yaml.safe_load(cfg_path.read_text())
cfg["data"]["batch_size"]  = BATCH_SIZE
cfg["data"]["num_workers"] = NUM_WORKERS
# Written back so meta.yaml records the config that actually trained, and so a
# resume compares equal: train.py raises if the checkpoint's config differs.
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

print(f"code staged -> {REPO}")
print(f"  batch_size={cfg['data']['batch_size']}  num_workers={cfg['data']['num_workers']}  "
      f"seed={cfg['seed']}  chunk_s={cfg['data']['chunk_s']}")
print(f"  loss: w={cfg['loss']['w']} w_m={cfg['loss']['w_m']} "
      f"tau_pres={cfg['loss']['tau_pres']} tau_abs={cfg['loss']['tau_abs']}")

if RESUME_FROM:
    dst = Path(OUT) / "model_mid.pt"
    shutil.copy2(RESUME_FROM, dst)
    ck = torch.load(dst, map_location="cpu", weights_only=False)
    print(f"resume: copied checkpoint from epoch {ck['epoch']}, best_val {ck['best_val']:.4f}")
    if ck.get("config") != cfg:
        raise SystemExit("checkpoint config != this config; train.py will refuse. "
                         "Restore the knobs used for that checkpoint.")

In [ ]:
# --- train ---------------------------------------------------------------
# Audio is read straight from the read-only data input; only outputs land
# in /kaggle/working.
#
# train.py prints one CSV row per epoch on STDOUT, with the same columns as
# history.csv. tqdm's progress bars go to STDERR. They are kept apart on
# purpose: the stdout block below is a valid history.csv, so if this session
# dies you can select it, paste it into a .csv, and lose nothing. history.csv
# itself is only written when training *finishes*.
#
# -u as well as train.py's own flush=True: stdout is a pipe here, and a killed
# session must not lose rows to a buffer.
cmd = [sys.executable, "-u", "scripts/train.py",
       "--split", "mid",
       "--epochs", str(EPOCHS),
       "--config", "experiments/configs/bsrnn_baseline.yaml",
       "--data-root", str(DATA / "data"),
       "--manifest-dir", str(DATA / "data/manifests"),
       "--outdir", OUT,
       "--results-dir", RES]
if RESUME_FROM:
    cmd.append("--resume")
print(" ".join(cmd), flush=True)

# Tee stdout: print each line live AND append it to disk, so the curve survives
# both ways. stderr is inherited, so the bars render normally and never mix into
# the copyable block.
live = Path(RES) / "history_live.csv"
import re
CSV_ROW = re.compile(r"^(epoch|\d+),")
with open(live, "a") as fh:
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                            stderr=None, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        if CSV_ROW.match(line):
            fh.write(line)
            fh.flush()
    rc = proc.wait()

print(f"\nexit {rc}   (rows also appended to {live})")
if rc != 0:
    raise SystemExit(f"training failed with exit {rc}")

In [ ]:
# --- what came out -------------------------------------------------------
import pandas as pd

ck = Path(OUT) / "model_mid.pt"
print(f"checkpoint: {ck}  ({ck.stat().st_size/1e6:.0f} MB)" if ck.exists() else "NO CHECKPOINT")

hist = Path(RES) / "history.csv"
live = Path(RES) / "history_live.csv"
# history.csv only exists if training finished; history_live.csv exists either way
src = hist if hist.exists() else live
if src.exists():
    h = pd.read_csv(src)
    h = h[h.epoch != "epoch"].astype({c: float for c in h.columns})  # drop repeated headers from a resume
    print(f"read {src.name} ({len(h)} epochs)")
    cols = ["epoch", "train_total", "val_total", "val_L_pres", "val_L_MR", "val_L_abs", "lr"]
    print(h[cols].to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
    best = h.loc[h.val_total.idxmin()]
    print(f"\nbest val_total {best.val_total:.4f} at epoch {int(best.epoch)}")
    if not hist.exists():
        print("NOTE: read from history_live.csv -- training did not finish, "
              "so meta.yaml and loss_plot.png were never written.")
    # The thing this run exists to answer. L_MR rising while val_total falls is
    # the enrolment-blind mute of 2026-08-25 reappearing at 940 speakers.
    d = h.val_L_MR.iloc[-1] - h.val_L_MR.iloc[0]
    print(f"val_L_MR {h.val_L_MR.iloc[0]:.4f} -> {h.val_L_MR.iloc[-1]:.4f}  ({d:+.4f})  "
          + ("STILL RISING: mute not escaped" if d > 0 else "falling: mute escaped"))
else:
    print("no history at all -- training died before finishing epoch 1. "
          "The CSV rows printed by the training cell are the only record; "
          "copy them into a .csv if you need them.")

rt = Path(REPO) / "docs/run_times.md"
if rt.exists():
    rows = [l for l in rt.read_text().splitlines() if l.startswith("| 20")]
    print("\nmeasured wall time (copy this row into the repo's docs/run_times.md):")
    for r in rows[:1]:
        print(" ", r)
print("\nDownload /kaggle/working/models/model_mid.pt and /kaggle/working/results/ "
      "before the session ends, or Save Version to keep them.")